In [19]:
import ast
import re
from collections import defaultdict

# def parse_file1(path):
#     data = {}
#     with open(path, 'r', encoding='utf-8') as f:
#         for line in f:
#             if '\t' not in line:
#                 continue
#             cat, raw = line.strip().split('\t', 1)
#             data[cat] = ast.literal_eval(raw)
#     return data
def parse_file1(path):
    data = {}
    with open(path, 'r', encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            parts = line.rstrip('\n').split('\t', 1)
            if len(parts) != 2:
                # Zeile nicht im erwarteten Format – überspringen oder warnen
                print(f"Warnung: Zeile {lineno} übersprungen: {line!r}")
                continue
            cat, raw = parts
            try:
                data[cat] = ast.literal_eval(raw)
            except Exception as e:
                print(f"Fehler beim Parsen in Zeile {lineno}: {e}")
    return data

def parse_file2(path):
    data = defaultdict(dict)
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    pattern = re.compile(r"\('([^']+)', \[(.*?)\]\)", re.DOTALL)
    matches = pattern.findall(content)

    for cat, raw_keywords in matches:
        # Format the string to be a list of tuples
        try:
            keyword_list = ast.literal_eval(f"[{raw_keywords}]")
            data[cat] = {kw: score for kw, score in keyword_list}
        except:
            continue
    return dict(data)


In [20]:
def compare_keyword_data_with_means(data1, data2):
    all_categories = set(data1) | set(data2)
    results = {}

    for cat in all_categories:
        kw1 = data1.get(cat, {})
        kw2 = data2.get(cat, {})

        keys1 = set(kw1)
        keys2 = set(kw2)

        common = keys1 & keys2
        only1 = keys1 - keys2
        only2 = keys2 - keys1

        diff_scores = {k: (kw1[k], kw2[k]) for k in common if abs(kw1[k] - kw2[k]) > 1e-5}

        if common:
            mean_file1 = sum(kw1[k] for k in common) / len(common)
            mean_file2 = sum(kw2[k] for k in common) / len(common)
            mean_abs_diff = sum(abs(kw1[k] - kw2[k]) for k in common) / len(common)
        else:
            mean_file1 = mean_file2 = mean_abs_diff = None

        results[cat] = {
            'common_keywords': sorted(common),
            'different_scores': diff_scores,
            'only_in_file1': sorted(only1),
            'only_in_file2': sorted(only2),
            'mean_file1': mean_file1,
            'mean_file2': mean_file2,
            'mean_abs_diff': mean_abs_diff,
        }
    return results


In [21]:
def show_comparison(results):
    for cat, diff in results.items():
        print(f"\n== Category: {cat} ==")
        print(f"Common keywords: {len(diff['common_keywords'])}")
        print(f"Keywords only in file 1: {diff['only_in_file1']}")
        print(f"Keywords only in file 2: {diff['only_in_file2']}")
        if diff['different_scores']:
            print("Keywords with different scores:")
            for k, (s1, s2) in diff['different_scores'].items():
                print(f"  {k}: file1={s1:.2f}, file2={s2:.2f}")
        if diff['mean_file1'] is not None:
            print(f"Mean score (file1): {diff['mean_file1']:.2f}")
            print(f"Mean score (file2): {diff['mean_file2']:.2f}")
            print(f"Mean abs. difference: {diff['mean_abs_diff']:.2f}")


In [ ]:
file1_data = parse_file1('~/2025_05_08_21_41_chisq_output.txt')
file2_data = parse_file2('~/output_rdd.txt')
file3_data = parse_file1('~/chisq_output.txt')
file4_data = parse_file2('~/output_rdd2.txt')
comparison = compare_keyword_data_with_means(file1_data, file2_data)
show_comparison(comparison)

Warnung: Zeile 23 übersprungen: "['acdelco', 'acne', 'acoustic', 'acre', 'acted', 'acting', 'action', 'actor', 'actors', 'actress', 'acura', 'adapter', 'addario', 'addicted', 'addicting', 'addictive', 'adjustment', 'adorable', 'ads', 'adventure', 'aftertaste', 'aired', 'airsoft', 'akai', 'albums', 'almonds', 'alpha', 'alternator', 'altima', 'ammo', 'amp', 'amplitube', 'android', 'animated', 'animation', 'anime', 'answering', 'antenna', 'ants', 'apos', 'appetite', 'apple', 'apply', 'apps', 'aquarium', 'ar', 'arch', 'arnley', 'aroma', 'arrangements', 'articulation', 'artisan', 'artist', 'artists', 'asus', 'atv', 'audio', 'author', 'authors', 'avent', 'avery', 'awesome', 'babies', 'back', 'backpacking', 'bag', 'bait', 'baking', 'ball', 'ballad', 'ballads', 'ballasts', 'balls', 'band', 'bands', 'banjo', 'barbie', 'barking', 'barrel', 'bass', 'bassinet', 'bath', 'bathroom', 'batteries', 'battery', 'bbq', 'beam', 'beard', 'bearings', 'bed', 'behringer', 'believable', 'berserker', 'bib', 'bic

In [23]:
comparison = compare_keyword_data_with_means(file3_data, file4_data)
show_comparison(comparison)


== Category: Health_and_Personal_Care ==
Common keywords: 75
Keywords only in file 1: []
Keywords only in file 2: []
Keywords with different scores:
  works: file1=156818.45, file2=156815.67
  dosage: file1=243680.72, file2=243654.67
  beard: file1=270406.95, file2=270391.07
  razor: file1=1135025.39, file2=1134944.44
  shaver: file1=966180.48, file2=966157.34
  stomach: file1=252295.87, file2=252288.59
  batteries: file1=149272.52, file2=149271.66
  detergent: file1=200697.45, file2=200689.62
  protein: file1=640750.79, file2=640730.95
  weight: file1=189160.68, file2=189158.17
  cleaning: file1=159793.05, file2=159787.29
  muscle: file1=164370.21, file2=164343.80
  appetite: file1=347931.04, file2=347827.59
  smell: file1=285861.67, file2=285853.54
  balm: file1=139194.54, file2=139188.13
  supplement: file1=578368.17, file2=578346.48
  capsule: file1=156161.06, file2=156138.09
  bottle: file1=401318.53, file2=401309.53
  swallow: file1=166143.51, file2=166124.68
  hairs: file1=2363